In [172]:
#.venv 
prod_connection_string = "DRIVER={ODBC Driver 17 for SQL Server};Server=CUBO-INTERMODA;Database=IMClientesIV;UID=iditm;PWD=Int3r-M0d@.Id@;Trusted_Connection=no;"
url = "https://unikfashiongt.odoo.com"
db = "rocketgithub-unikfashiongt-odoo-sh-main-25251833"
username = "rmartinez@intermoda.com.hn"
password = "Intermod@2026/?"

#Autenticación con Odoo
from datetime import datetime
import xmlrpc.client
import pandas as pd
import pyodbc
import json


common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
uid = common.authenticate(db, username, password, {})

In [167]:
#Obtener el producto por su código de barras
models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object")
products = pd.DataFrame(models.execute_kw(db, uid, password, 'product.product', 'search_read', [[['barcode', '!=', False]]] ,{'limit': 1})).rename(columns={'barcode': 'CodigoBarra'})
print(products)
#print(products[['CodigoBarra','qty_available']])

      id activity_ids  activity_state  activity_user_id  activity_type_id  \
0  14928           []           False             False             False   

   activity_type_icon  activity_date_deadline  my_activity_date_deadline  \
0               False                   False                      False   

   activity_summary  activity_exception_decoration  ...  can_be_expensed  \
0             False                          False  ...            False   

   purchase_method purchase_line_warn purchase_line_warn_msg lot_valuated  \
0          receive         no-message                  False        False   

   available_in_pos  to_weight  pos_categ_ids  public_description  \
0              True      False           [34]               False   

   property_account_creditor_price_difference  
0                                       False  

[1 rows x 166 columns]


In [168]:
#obtener los clientes
with pyodbc.connect(prod_connection_string) as conn:
    cursor = conn.cursor()
    cursor.execute("EXEC dbo.SP_ObtenerClientes")

    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()

    df = pd.DataFrame.from_records(rows, columns=columns)

    #print(df[df['CodigoCliente'] == "IMGT-000001134"])
    conn.commit()
Clientes = df[df['CodigoCliente'] == "IMGT-000001134"]
print(Clientes)


                                           Referencia   CodigoCliente Empresa  \
593  (IMGT-000001134)  UNIK FASHION, SOCIEDAD ANONIMA  IMGT-000001134    IMGT   

                            Cliente  
593  UNIK FASHION, SOCIEDAD ANONIMA  


In [169]:
#obtener las Tiendas
with pyodbc.connect(prod_connection_string) as conn:
    cursor = conn.cursor()
    cursor.execute("EXEC dbo.SP_ObtenerTiendas ?", ("IMGT-000001134",))
    
    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()

    df = pd.DataFrame.from_records(rows, columns=columns)

    #print(df)
    conn.commit()
Tiendas = df
print(Tiendas)

   CodigoTienda                         Tienda
0          None                Sucursal Zacapa
1          None               Sucursal Utzuleu
2          None                 Sucursal Peten
3          None         Sucursal Huehuetenango
4          None               Sucursal Jutiapa
5          None             Sucursal Escuintla
6          None              Sucursal Portales
7          None          Sucursal Portales Old
8          None        Sucursal Puerto Barrios
9          None            Sucursal Retalhuleu
10         None                Sucursal Poptun
11         None            Sucursal Coatepeque
12         None         Sucursal Chimaltenango
13         None             Sucursal Malacatan
14         None                          COBAN
15         None           Sucursal Metrocentro
16         None            Sucursal Interplaza
17         None           Tienda Huehuetenango
18         None           Sucursal Mazatenango
19         None  Sucursal Interplaza Escuintla


In [175]:
#obtener la información del producto por su código de barras
import string


for index, row in products.iterrows():
    #print(row)

    with pyodbc.connect(prod_connection_string) as conn:
        cursor = conn.cursor()
        query = "EXEC dbo.SP_GetCodigosDeBarraInfo ?;"
    
        json_data = row[['CodigoBarra']].to_json(orient='index', force_ascii=False)
            
        cursor.execute(query, json_data)
        columns = [column[0]  for column in cursor.description]
        rows = cursor.fetchall()
        results = pd.DataFrame.from_records(rows, columns=columns)
        #print(row.to_json(orient='index', force_ascii=False))

        hoy = datetime.strptime(row["create_date"], "%Y-%m-%d %H:%M:%S")

        json_data = {
            "CodigoCliente": Clientes['CodigoCliente'].values[0],
            "Cliente": Clientes['Cliente'].values[0],
            "CodigoTienda": None,
            "Tienda": None,
            "FechaCreacion": hoy.strftime("%Y-%m-%d"),
            "año": hoy.year,
            "NoMes": hoy.month,
            "Mes": hoy.strftime("%B"),
            "dia": hoy.day,
            "CodigoBarra": row['CodigoBarra'],
            "CodigoArticulo": results['CodigoArticulo'].values[0],
            "Descripcion": results['Descripcion'].values[0],
            "CodigoColor": results['CodigoColor'].values[0],
            "Color": results['Color'].values[0],
            "Talla": results['Talla'].values[0],
            "linea": results['Linea'].values[0],
            "Sublinea": results['Sublinea'].values[0],
            "Categoria": results['Categoria'].values[0],
            "Base": results['Base'].values[0],
            "Genero": results['Genero'].values[0],
            "Clasificacion": results['ClasificacionAX'].values[0],
            "LoteOrigen": results['LoteOrigen'].values[0],
            "PedidoVenta": None,
            "FechaFactura": None,
            "Costo": None,
            "Precio": row['list_price'],
            "Cantidad": row['qty_available'],
            "CostoTotal": None,
        }

        

        print(json_data)


{'CodigoCliente': 'IMGT-000001134', 'Cliente': 'UNIK FASHION, SOCIEDAD ANONIMA', 'CodigoTienda': None, 'Tienda': None, 'FechaCreacion': '2026-03-19', 'año': 2026, 'NoMes': 3, 'Mes': 'March', 'dia': 19, 'CodigoBarra': '7424625714570', 'CodigoArticulo': '10 11 01 03 897 0001', 'Descripcion': 'BERMUDA HOMBRE STRAIGHT MEDIUM RISE', 'CodigoColor': 'T2', 'Color': 'INDIGO MEDIO', 'Talla': '29', 'linea': 'DENIM', 'Sublinea': 'DENIM', 'Categoria': 'MODA', 'Base': 'VI897', 'Genero': 'Hombre', 'Clasificacion': 'PRIMERAS', 'LoteOrigen': '622F', 'PedidoVenta': None, 'FechaFactura': None, 'Costo': None, 'Precio': 265.0, 'Cantidad': 1.0, 'CostoTotal': None}
